# 04 - Staker Revenue Flows And Realization Quality

This notebook focuses on payout attribution quality and realization, using deterministic
evidence artifacts generated by `src/analysis/evidence.py`.

**Source classes**
- ONCHAIN_EXECUTED-derived artifacts (via evidence builder)
- GOVERNANCE_FORUM references embedded in cycle payloads


## Workflow

1. Optionally refresh claims + evidence artifacts (requires RPC access).
2. Load latest evidence directory.
3. Visualize deposited vs claimed-attributed vs residual.
4. Quantify realization ratio and source-of-funds coverage.


In [ ]:
from pathlib import Path
import sys
import json
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")

cwd = Path.cwd().resolve()
root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "notebooks" / "notebook_utils.py").exists():
        root = candidate
        break
if root is None:
    raise RuntimeError("Could not locate repository root containing notebooks/notebook_utils.py")

if root.as_posix() not in sys.path:
    sys.path.insert(0, root.as_posix())

from notebooks import notebook_utils as nbx
ROOT = nbx.setup_notebook_context()
print(f"Project root: {ROOT}")
print(f"Notebook run started (UTC): {nbx.utc_now_iso()}")


In [ ]:
RUN_REFRESH_PIPELINE = False
EVIDENCE_DIR_OVERRIDE = None  # Example: ROOT / "results/proofs/evidence_2026-02-09-independent"

if RUN_REFRESH_PIPELINE:
    nbx.run_make_targets(["refresh_claims", "refresh_lvusdc_nav", "evidence"])

if EVIDENCE_DIR_OVERRIDE is not None:
    evidence_dir = Path(EVIDENCE_DIR_OVERRIDE)
else:
    evidence_dir = nbx.latest_evidence_dir()

if evidence_dir is None:
    raise FileNotFoundError("No evidence directory found under results/proofs/evidence_*")

print("Using evidence dir:", evidence_dir)


In [ ]:
payout_summary = nbx.load_json(evidence_dir / "payout_attribution_summary.json")
sof_summary = nbx.load_json(evidence_dir / "source_of_funds_summary.json")
revenue_summary = nbx.load_json(evidence_dir / "staker_revenue_canonical_summary.json")

campaigns = pd.DataFrame(payout_summary.get("campaigns") or [])
if campaigns.empty:
    raise RuntimeError("No campaigns found in payout attribution summary.")

view_cols = [
    "label",
    "reward_token_symbol",
    "staker_revenue_deposited_tokens",
    "staker_revenue_claimed_tokens_attributed",
    "staker_revenue_unclaimed_tokens_residual",
    "residual_ratio",
    "attribution_confidence_class",
]

display(campaigns[view_cols])


In [ ]:
plot_df = campaigns.copy()
plot_df = plot_df.sort_values("label")

x = np.arange(len(plot_df))
claimed = plot_df["staker_revenue_claimed_tokens_attributed"].astype(float).values
residual = plot_df["staker_revenue_unclaimed_tokens_residual"].astype(float).values

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

axes[0].bar(x, claimed, label="Claimed attributed", color="#2a9d8f")
axes[0].bar(x, residual, bottom=claimed, label="Residual", color="#e76f51")
axes[0].set_xticks(x, plot_df["label"])
axes[0].set_title("Campaign Realization Stack")
axes[0].set_ylabel("Reward token units")
axes[0].legend()

axes[1].bar(plot_df["label"], plot_df["residual_ratio"].astype(float) * 100.0, color="#6d597a")
axes[1].set_title("Residual Ratio By Campaign")
axes[1].set_ylabel("Residual %")
axes[1].axhline(5.0, color="#d62828", linestyle="--", linewidth=1.5, label="5% gate threshold")
axes[1].legend()

plt.show()


In [ ]:
agg = revenue_summary.get("aggregate") or {}
deposited_total = float(agg.get("staker_revenue_deposited_tokens_total") or 0.0)
claimed_total = float(agg.get("staker_revenue_claimed_tokens_attributed_total") or 0.0)
realization_ratio = (claimed_total / deposited_total) if deposited_total > 0 else np.nan

sof = (sof_summary.get("source_of_funds_comparison") or {})
coverage_ratio = sof.get("comparable_coverage_ratio")
sof_status = sof.get("status")

summary = pd.DataFrame([
    {
        "deposited_total": deposited_total,
        "claimed_attributed_total": claimed_total,
        "realization_ratio": realization_ratio,
        "source_of_funds_status": sof_status,
        "source_of_funds_coverage_ratio": coverage_ratio,
    }
])
display(summary)

display(Markdown(
    f"""
**Interpretation**
- Aggregate realization ratio: `{nbx.fmt_pct(realization_ratio)}`
- Source-of-funds status: `{sof_status}`
- Comparable coverage ratio: `{coverage_ratio:.3f}x` if available

Treat realization as bounded unless campaign-exact attribution can be proven.
"""
))
